# Model 1 v3 – 07 Train + in-source evaluation
**Purpose:** Retrain the same two models as v2 (MobileNetV3-Large, DeiT-Small) on the v3 dataset
(= v2 + photos of people, limb close-ups and hands as NON_XRAY). Training code and settings are IDENTICAL to v2
(notebook 04); only the data changed. Then compare v2 vs v3 on the v3 test set.
**Inputs:** outputs/model1_v3/manifest_split_v3.csv, cache_224_v3.npz, outputs/model1_v2/ (v2 models)
**Outputs:** outputs/model1_v3/{name}_model1v3_best.pt, preds_*.csv, info_*.json
**Selection:** best epoch = lowest validation loss (same as v2).

In [11]:
# ===== SETTINGS =====
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/Research')
OUT_DIR    = DRIVE_ROOT / 'outputs' / 'model1_v3'
MANIFEST   = OUT_DIR / 'manifest_split_v3.csv'
CACHE      = OUT_DIR / 'cache_224_v3.npz'
V2_DIR     = DRIVE_ROOT / 'outputs' / 'model1_v2'

SEED, IMG_SIZE, BATCH_SIZE = 42, 224, 64
EPOCHS, LR, PATIENCE = 10, 1e-4, 3
P_LOWRES, P_JPEG = 0.3, 0.3
MODELS = {'mobilenetv3': 'torchvision', 'deit_small': 'deit_small_patch16_224.fb_in1k'}

In [12]:
!pip install -q timm
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd, torch, random, timm, time, os, io, json
device = 'cuda'
df = pd.read_csv(MANIFEST)
cache = np.load(CACHE, allow_pickle=True)
X = cache['images']
assert (cache['paths'][df.cache_idx] == df.path).all(), 'Manifest and cache do not match!'
splits = {s: df[df.split == s].reset_index(drop=True) for s in ['train', 'val', 'test']}
print({s: len(f) for s, f in splits.items()}, '| cache', X.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{'train': 7136, 'val': 1498, 'test': 1503} | cache (10285, 224, 224)


## Step 1 – Data loaders and augmentation (unchanged from v2)

In [13]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

def degrade(img):
    if random.random() < P_LOWRES:
        s = random.randint(64, 160)
        img = img.resize((s, s), Image.BILINEAR).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    if random.random() < P_JPEG:
        buf = io.BytesIO(); img.save(buf, format='JPEG', quality=random.randint(30, 90))
        img = Image.open(buf).convert('L')
    return img

def make_loaders(mean, std):
    to_model = [T.ToTensor(), T.Lambda(lambda t: t.repeat(3, 1, 1)), T.Normalize(mean, std)]
    train_tf = T.Compose([T.RandomResizedCrop(IMG_SIZE, scale=(0.75, 1.0), ratio=(0.9, 1.1)),
                          T.RandomHorizontalFlip(), T.RandomRotation(20),
                          T.ColorJitter(brightness=0.2, contrast=0.2), *to_model])
    eval_tf = T.Compose(to_model)
    class DS(Dataset):
        def __init__(self, f, train): self.idx, self.y, self.train = f.cache_idx.values, f.y.values, train
        def __len__(self): return len(self.idx)
        def __getitem__(self, i):
            img = Image.fromarray(X[self.idx[i]])
            return (train_tf(degrade(img)) if self.train else eval_tf(img)), self.y[i]
    return {s: DataLoader(DS(splits[s], s == 'train'), batch_size=BATCH_SIZE, shuffle=(s == 'train'),
                          num_workers=2, pin_memory=True) for s in splits}

## Step 2 – Model, metrics and training loop (unchanged from v2, only the file name says v3)

In [14]:
import torch.nn as nn, torch.nn.functional as F
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights
from sklearn.metrics import roc_auc_score

def build(name):
    if MODELS[name] == 'torchvision':
        m = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.IMAGENET1K_V2)
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, 2)
        return m, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
    m = timm.create_model(MODELS[name], pretrained=True, num_classes=2)
    return m, list(m.pretrained_cfg['mean']), list(m.pretrained_cfg['std'])

def metrics(p, y):
    pred = (p >= 0.5).astype(int)
    return {'acc': (pred == y).mean(), 'auc': roc_auc_score(y, p),
            'FAR': ((pred == 1) & (y == 0)).sum() / (y == 0).sum(),     # non-X-ray accepted
            'FRR': ((pred == 0) & (y == 1)).sum() / (y == 1).sum()}     # X-ray rejected

counts = splits['train'].y.value_counts().sort_index().values
w = (counts.sum() / (2 * counts)) ** 0.5; CLASS_W = torch.tensor(w / w.mean(), dtype=torch.float32)

def save_preds(name, split, p):
    out = splits[split][['path', 'source', 'label', 'y']].copy(); out['p_xray'] = p
    out.to_csv(OUT_DIR / f'preds_{name}_{split}.csv', index=False)

def train_and_evaluate(name):
    torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
    model, mean, std = build(name); model = model.to(device)
    loaders = make_loaders(mean, std)
    crit = nn.CrossEntropyLoss(weight=CLASS_W.to(device))
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    scaler = torch.amp.GradScaler('cuda'); path = OUT_DIR / f'{name}_model1v3_best.pt'

    @torch.no_grad()
    def predict(loader):
        model.eval(); ps = []
        for xb, _ in loader:
            with torch.amp.autocast('cuda'):
                ps.append(F.softmax(model(xb.to(device)).float(), 1)[:, 1].cpu())
        return torch.cat(ps).numpy()

    best, best_ep, wait = 1e9, 0, 0
    for ep in range(1, EPOCHS + 1):
        t0 = time.time(); model.train(); tot = 0.0
        for xb, yb in loaders['train']:
            xb, yb = xb.to(device), yb.to(device); opt.zero_grad()
            with torch.amp.autocast('cuda'):
                loss = crit(model(xb), yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); tot += loss.item() * len(yb)
        sched.step()
        p = predict(loaders['val']); y = splits['val'].y.values
        vloss = F.binary_cross_entropy(torch.tensor(p).clamp(1e-6, 1 - 1e-6), torch.tensor(y, dtype=torch.float32)).item()
        m = metrics(p, y)
        print(f'Epoch {ep:2d} | train loss {tot/len(splits["train"]):.4f} | val loss {vloss:.4f} | val acc {m["acc"]:.4f} '
              f'| FAR {m["FAR"]:.3f} | FRR {m["FRR"]:.3f} | {time.time()-t0:.0f}s')
        if vloss < best:
            best, best_ep, wait = vloss, ep, 0
            torch.save({'state_dict': model.state_dict(), 'name': name, 'timm_name': MODELS[name],
                        'mean': mean, 'std': std, 'epoch': ep}, path); print('   ✓ saved best model')
        else:
            wait += 1
            if wait >= PATIENCE: print('Early stopping.'); break

    model.load_state_dict(torch.load(path, map_location=device)['state_dict'])
    for s in ['val', 'test']: save_preds(name, s, predict(loaders[s]))
    x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device); model.eval()
    with torch.no_grad():
        for _ in range(10): model(x)
        torch.cuda.synchronize(); t0 = time.time()
        for _ in range(100): model(x)
        torch.cuda.synchronize()
    info = {'model': name, 'best_epoch': best_ep, 'params_M': round(sum(q.numel() for q in model.parameters()) / 1e6, 1),
            'size_mb': round(os.path.getsize(path) / 1e6, 1), 'ms_per_img': round((time.time() - t0) * 10, 1)}
    json.dump(info, open(OUT_DIR / f'info_{name}.json', 'w')); print('Done:', info)
    del model; torch.cuda.empty_cache()

In [15]:
train_and_evaluate('mobilenetv3')

Epoch  1 | train loss 0.1092 | val loss 0.3530 | val acc 0.8738 | FAR 0.256 | FRR 0.000 | 31s
   ✓ saved best model
Epoch  2 | train loss 0.0151 | val loss 0.0423 | val acc 0.9860 | FAR 0.028 | FRR 0.000 | 27s
   ✓ saved best model
Epoch  3 | train loss 0.0095 | val loss 0.0098 | val acc 0.9960 | FAR 0.005 | FRR 0.003 | 27s
   ✓ saved best model
Epoch  4 | train loss 0.0088 | val loss 0.0044 | val acc 0.9993 | FAR 0.001 | FRR 0.000 | 30s
   ✓ saved best model
Epoch  5 | train loss 0.0043 | val loss 0.0053 | val acc 0.9980 | FAR 0.003 | FRR 0.001 | 27s
Epoch  6 | train loss 0.0069 | val loss 0.0021 | val acc 0.9993 | FAR 0.000 | FRR 0.001 | 26s
   ✓ saved best model
Epoch  7 | train loss 0.0040 | val loss 0.0016 | val acc 0.9993 | FAR 0.001 | FRR 0.000 | 27s
   ✓ saved best model
Epoch  8 | train loss 0.0030 | val loss 0.0013 | val acc 1.0000 | FAR 0.000 | FRR 0.000 | 27s
   ✓ saved best model
Epoch  9 | train loss 0.0022 | val loss 0.0010 | val acc 1.0000 | FAR 0.000 | FRR 0.000 | 27s


In [16]:
train_and_evaluate('deit_small')

model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch  1 | train loss 0.0519 | val loss 0.0121 | val acc 0.9940 | FAR 0.009 | FRR 0.003 | 29s
   ✓ saved best model
Epoch  2 | train loss 0.0127 | val loss 0.0055 | val acc 0.9980 | FAR 0.004 | FRR 0.000 | 30s
   ✓ saved best model
Epoch  3 | train loss 0.0149 | val loss 0.0017 | val acc 1.0000 | FAR 0.000 | FRR 0.000 | 36s
   ✓ saved best model
Epoch  4 | train loss 0.0073 | val loss 0.0012 | val acc 1.0000 | FAR 0.000 | FRR 0.000 | 30s
   ✓ saved best model
Epoch  5 | train loss 0.0055 | val loss 0.0027 | val acc 0.9987 | FAR 0.003 | FRR 0.000 | 30s
Epoch  6 | train loss 0.0013 | val loss 0.0008 | val acc 0.9993 | FAR 0.000 | FRR 0.001 | 28s
   ✓ saved best model
Epoch  7 | train loss 0.0014 | val loss 0.0045 | val acc 0.9973 | FAR 0.000 | FRR 0.005 | 30s
Epoch  8 | train loss 0.0014 | val loss 0.0004 | val acc 1.0000 | FAR 0.000 | FRR 0.000 | 28s
   ✓ saved best model
Epoch  9 | train loss 0.0004 | val loss 0.0004 | val acc 1.0000 | FAR 0.000 | FRR 0.000 | 30s
Epoch 10 | train loss 

## Step 3 – v2 vs v3 on the v3 test set
The v3 test set = the v2 test set + test images of the new sources. v2 never saw photos of people/hands,
so this shows how often v2 accepted them as X-rays, and whether v3 still accepts all real X-rays.

In [17]:
from torchvision.models import mobilenet_v3_large

def load_ck(path):
    ck = torch.load(path, map_location=device)
    if ck['timm_name'] == 'torchvision':
        m = mobilenet_v3_large(); m.classifier[3] = nn.Linear(m.classifier[3].in_features, 2)
    else:
        m = timm.create_model(ck['timm_name'], pretrained=False, num_classes=2)
    m.load_state_dict(ck['state_dict']); return m.eval().to(device), ck['mean'], ck['std']

@torch.no_grad()
def p_xray(model, mean, std, arr):
    mean = torch.tensor(mean).view(1, 3, 1, 1).to(device); std = torch.tensor(std).view(1, 3, 1, 1).to(device)
    out = []
    for k in range(0, len(arr), 64):
        t = torch.from_numpy(arr[k:k+64]).float().div(255).unsqueeze(1).repeat(1, 3, 1, 1).to(device)
        out.append(torch.softmax(model((t - mean) / std), 1)[:, 1].cpu().numpy())
    return np.concatenate(out)

imgs = X[splits['test'].cache_idx.values]; y = splits['test'].y.values
P = {'v2 MobileNetV3': p_xray(*load_ck(V2_DIR / 'mobilenetv3_model1v2_best.pt'), imgs),
     'v2 DeiT-Small':  p_xray(*load_ck(V2_DIR / 'deit_small_model1v2_best.pt'), imgs)}
P['v2 Ensemble'] = (P['v2 MobileNetV3'] + P['v2 DeiT-Small']) / 2
P['v3 MobileNetV3'] = pd.read_csv(OUT_DIR / 'preds_mobilenetv3_test.csv').p_xray.values
P['v3 DeiT-Small']  = pd.read_csv(OUT_DIR / 'preds_deit_small_test.csv').p_xray.values
P['v3 Ensemble'] = (P['v3 MobileNetV3'] + P['v3 DeiT-Small']) / 2
print(pd.DataFrame({k: metrics(v, y) for k, v in P.items()}).T.round(4).to_string())

print('\nErrors per source (test, threshold 0.5):')
t = splits['test'][['source', 'label']].copy()
for k, v in P.items(): t[k] = ((v >= 0.5).astype(int) != y).astype(int)
err = t.groupby(['label', 'source'])[list(P)].sum(); err.insert(0, 'n', t.groupby(['label', 'source']).size())
print(err[err[list(P)].sum(1) > 0].to_string())

                   acc     auc     FAR     FRR
v2 MobileNetV3  0.9854  0.9999  0.0285  0.0013
v2 DeiT-Small   0.9880  0.9999  0.0244  0.0000
v2 Ensemble     0.9887  0.9999  0.0230  0.0000
v3 MobileNetV3  0.9993  1.0000  0.0000  0.0013
v3 DeiT-Small   1.0000  1.0000  0.0000  0.0000
v3 Ensemble     1.0000  1.0000  0.0000  0.0000

Errors per source (test, threshold 0.5):
                                n  v2 MobileNetV3  v2 DeiT-Small  v2 Ensemble  v3 MobileNetV3  v3 DeiT-Small  v3 Ensemble
label    source                                                                                                          
NON_XRAY new_Hands_photos      90              14             15           14               0              0            0
         new_People_HAR        90               2              1            1               0              0            0
         new_People_crops_HAR  60               5              2            2               0              0            0
X_RAY    m2_Hip_to_

## Summary – in-source test (v3 test set, 1,503 images)
| Model | Acc | FAR | FRR |
|---|---|---|---|
| v2 Ensemble | 98.9% | 2.3% | 0% |
| v3 MobileNetV3 | 99.9% | 0% | 0.1% |
| v3 DeiT-Small | 100% | 0% | 0% |
| v3 Ensemble | 100% | 0% | 0% |
v2 accepted 14/90 hand photos, 1/90 people photos, 2/60 limb close-ups as X-rays; v3 accepts none and still accepts all X-rays.
Training: same code/settings as v2. MobileNetV3 best epoch 9 (reproduced exactly on re-run); DeiT best epoch 8
(saved run; DeiT varied slightly between GPU runs). Caution: same-source test – generalisation checked in notebook 08.